# MobileNetV2 From Scratch

Notebook này train riêng MobileNetV2 từ đầu (`weights=None`), không fine-tune hai giai đoạn.

Pipeline được viết lại đơn giản: đọc data, split, tạo `tf.data`, visualize, train, evaluate và lưu artifact.


## 1. Setup


In [ ]:
from pathlib import Path
import json
import math
import random
import shutil
import time
import warnings
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

MODEL_NAME = 'MobileNetV2'
FILE_PREFIX = 'mobilenetv2_simple'

REQUIRED_CLASSES = ['bicycle', 'boat', 'bus', 'car', 'helicopter', 'minibus', 'motorcycle', 'train', 'truck']
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
IMG_SIZE = (224, 224)
PER_REPLICA_BATCH_SIZE = 32

EPOCHS = 80
LEARNING_RATE = 2e-4
DROPOUT_RATE = 0.40
WEIGHT_DECAY = 2e-4
LABEL_SMOOTHING = 0.08
BN_MOMENTUM = 0.90
LR_WARMUP_EPOCHS = 5
LR_MIN_FACTOR = 0.01
EARLY_STOPPING_PATIENCE = 12
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
MAX_IMAGES_PER_CLASS = None
RUN_TRAINING = True

GPU_DEVICES = tf.config.list_physical_devices('GPU')
for gpu in GPU_DEVICES:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as exc:
        print('Cannot set GPU memory growth:', exc)

strategy = tf.distribute.MirroredStrategy() if len(GPU_DEVICES) > 1 else tf.distribute.get_strategy()
NUM_REPLICAS = strategy.num_replicas_in_sync
BATCH_SIZE = PER_REPLICA_BATCH_SIZE * NUM_REPLICAS
tf.keras.mixed_precision.set_global_policy('float32')

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
ARTIFACT_DIR = OUTPUT_DIR / f'{FILE_PREFIX}_artifacts'
MODEL_DIR = ARTIFACT_DIR / 'models'
FIGURE_DIR = ARTIFACT_DIR / 'figures'
for directory in [OUTPUT_DIR, ARTIFACT_DIR, MODEL_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print('TensorFlow:', tf.__version__)
print('GPUs:', GPU_DEVICES)
print('Strategy:', type(strategy).__name__, '| replicas:', NUM_REPLICAS)
print('Batch size:', BATCH_SIZE)
print('Model:', MODEL_NAME, '| epochs:', EPOCHS, '| lr:', LEARNING_RATE, '| wd:', WEIGHT_DECAY)
print('Artifacts:', ARTIFACT_DIR)



## 2. Load Dataset


In [ ]:
DATA_ROOT = Path('/kaggle/input/datasets/leighk/vehicle-dataset/cleaned')
DATA_MODE = 'cleaned'

if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Dataset folder not found: {DATA_ROOT}')


def scan_cleaned_dataset(root):
    rows = []
    for label_dir in sorted([p for p in Path(root).iterdir() if p.is_dir()]):
        image_paths = [p for p in sorted(label_dir.rglob('*')) if p.suffix.lower() in IMAGE_EXTENSIONS]
        if MAX_IMAGES_PER_CLASS is not None:
            image_paths = image_paths[:MAX_IMAGES_PER_CLASS]
        for path in image_paths:
            rows.append({'image_path': str(path), 'label': label_dir.name, 'split': None})
    return pd.DataFrame(rows)


df = scan_cleaned_dataset(DATA_ROOT)
if df.empty:
    raise ValueError('No images found.')

available = sorted(df['label'].unique())
class_names = [c for c in REQUIRED_CLASSES if c in available] + [c for c in available if c not in REQUIRED_CLASSES]
label_to_id = {label: idx for idx, label in enumerate(class_names)}
id_to_label = {idx: label for label, idx in label_to_id.items()}
num_classes = len(class_names)

df = df[df['label'].isin(class_names)].copy()
df['label_id'] = df['label'].map(label_to_id).astype('int32')

print('DATA_ROOT:', DATA_ROOT)
print('Classes:', class_names)
print('Total images:', len(df))
display(df.head())


## 3. Split Dataset


In [ ]:
if DATA_MODE == 'splits' and df['split'].notna().all():
    split_df = df.copy()
    split_df['split'] = split_df['split'].map(normalized_split)
    train_df = split_df[split_df['split'] == 'train'].copy()
    val_df = split_df[split_df['split'] == 'val'].copy()
    test_df = split_df[split_df['split'] == 'test'].copy()
else:
    if abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) > 1e-6:
        raise ValueError('TRAIN_RATIO + VAL_RATIO + TEST_RATIO must equal 1.')
    train_val_df, test_df = train_test_split(
        df, test_size=TEST_RATIO, stratify=df['label_id'], random_state=SEED
    )
    val_relative = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
    train_df, val_df = train_test_split(
        train_val_df, test_size=val_relative, stratify=train_val_df['label_id'], random_state=SEED
    )
    train_df = train_df.copy(); train_df['split'] = 'train'
    val_df = val_df.copy(); val_df['split'] = 'val'
    test_df = test_df.copy(); test_df['split'] = 'test'
    split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

split_df.to_csv(ARTIFACT_DIR / 'split_dataframe.csv', index=False)

print('Train:', len(train_df), '| Val:', len(val_df), '| Test:', len(test_df))
display(split_df.groupby(['split', 'label']).size().unstack(fill_value=0).reindex(columns=class_names))

train_paths = set(train_df['image_path'])
val_paths = set(val_df['image_path'])
test_paths = set(test_df['image_path'])
print('Train/val overlap:', len(train_paths & val_paths))
print('Train/test overlap:', len(train_paths & test_paths))
print('Val/test overlap:', len(val_paths & test_paths))


## 4. Dataset Visualization


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.countplot(data=df, x='label', order=class_names, ax=axes[0])
axes[0].set_title('Class distribution')
axes[0].tick_params(axis='x', rotation=35)

split_counts = split_df.groupby(['split', 'label']).size().reset_index(name='count')
sns.barplot(data=split_counts, x='label', y='count', hue='split', order=class_names, ax=axes[1])
axes[1].set_title('Train / val / test distribution')
axes[1].tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()


def show_samples(dataframe, samples_per_class=3):
    selected = []
    for label in class_names:
        part = dataframe[dataframe['label'] == label]
        if not part.empty:
            selected.append(part.sample(min(samples_per_class, len(part)), random_state=SEED))
    sample_df = pd.concat(selected, ignore_index=True)
    cols = samples_per_class
    rows = len(class_names)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 2.5))
    axes = np.array(axes).reshape(rows, cols)
    for r, label in enumerate(class_names):
        part = sample_df[sample_df['label'] == label].reset_index(drop=True)
        for c in range(cols):
            ax = axes[r, c]
            ax.axis('off')
            if c < len(part):
                img = Image.open(part.loc[c, 'image_path']).convert('RGB')
                ax.imshow(img)
                ax.set_title(label, fontsize=9)
    plt.tight_layout()
    plt.show()


show_samples(train_df, samples_per_class=3)


## 5. tf.data Pipeline


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def load_image(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label


def make_dataset(dataframe, training=False):
    paths = dataframe['image_path'].astype(str).values
    labels = dataframe['label_id'].astype('int32').values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(dataframe), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    return ds.prefetch(AUTOTUNE)


train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)


def inspect_dataset(ds, name):
    images, labels = next(iter(ds))
    min_value = float(tf.reduce_min(images).numpy())
    max_value = float(tf.reduce_max(images).numpy())
    print(f'{name}: images {images.shape} {images.dtype}, range=({min_value:.3f}, {max_value:.3f})')
    print(f'{name}: labels {labels.shape} {labels.dtype}, sample={labels[:10].numpy()}')
    assert min_value >= 0.0 and max_value <= 1.01, f'{name} image range must be [0, 1], got ({min_value:.3f}, {max_value:.3f})'


inspect_dataset(train_ds, 'train_ds')
inspect_dataset(val_ds, 'val_ds')
inspect_dataset(test_ds, 'test_ds')

label_map = {
    'label_to_id': label_to_id,
    'id_to_label': {str(k): v for k, v in id_to_label.items()},
}
(ARTIFACT_DIR / 'label_map.json').write_text(json.dumps(label_map, ensure_ascii=False, indent=2), encoding='utf-8')


## 6. Feature Visualization


In [ ]:
data_augmentation_preview = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomTranslation(0.06, 0.06),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.12),
    tf.keras.layers.RandomContrast(0.12),
])

sample_row = train_df.sample(1, random_state=SEED).iloc[0]
original = Image.open(sample_row['image_path']).convert('RGB')
resized = original.resize(IMG_SIZE)
normalized = np.asarray(resized).astype('float32') / 255.0
augmented = data_augmentation_preview(tf.expand_dims(normalized, axis=0), training=True)[0].numpy()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(original); axes[0].set_title(f'Original {original.size}')
axes[1].imshow(resized); axes[1].set_title('Resized 224x224')
axes[2].hist(normalized.ravel(), bins=40); axes[2].set_title('Pixel values [0, 1]')
axes[3].imshow(np.clip(augmented, 0, 1)); axes[3].set_title('Augmentation preview')
for ax in [axes[0], axes[1], axes[3]]:
    ax.axis('off')
plt.tight_layout()
plt.show()


def build_visualization_features(dataframe, max_per_class=80):
    sampled = []
    for label in class_names:
        part = dataframe[dataframe['label'] == label]
        if not part.empty:
            sampled.append(part.sample(min(max_per_class, len(part)), random_state=SEED))
    sample_df = pd.concat(sampled, ignore_index=True)

    features = []
    labels = []
    for _, row in sample_df.iterrows():
        img = Image.open(row['image_path']).convert('RGB').resize((32, 32))
        arr = np.asarray(img).astype('float32') / 255.0
        features.append(arr.ravel())
        labels.append(row['label'])
    return np.asarray(features), np.asarray(labels)


X_vis, y_vis = build_visualization_features(df, max_per_class=80)
X_scaled = StandardScaler().fit_transform(X_vis)

pca = PCA(n_components=2, random_state=SEED)
pca_coords = pca.fit_transform(X_scaled)
pca_df = pd.DataFrame({'pc1': pca_coords[:, 0], 'pc2': pca_coords[:, 1], 'label': y_vis})
plt.figure(figsize=(9, 7))
sns.scatterplot(data=pca_df, x='pc1', y='pc2', hue='label', s=35, alpha=0.85)
plt.title('PCA visualization of pixel features')
plt.tight_layout()
plt.show()

if len(X_scaled) >= 50:
    n_vis = min(len(X_scaled), 800)
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(X_scaled), size=n_vis, replace=False)
    perplexity = min(30, max(5, n_vis // 10))
    tsne = TSNE(n_components=2, perplexity=perplexity, init='pca', learning_rate='auto', random_state=SEED)
    tsne_coords = tsne.fit_transform(X_scaled[idx])
    tsne_df = pd.DataFrame({'x': tsne_coords[:, 0], 'y': tsne_coords[:, 1], 'label': y_vis[idx]})
    plt.figure(figsize=(9, 7))
    sns.scatterplot(data=tsne_df, x='x', y='y', hue='label', s=35, alpha=0.85)
    plt.title('t-SNE visualization of pixel features')
    plt.tight_layout()
    plt.show()


## 7. Model


In [ ]:
@tf.keras.utils.register_keras_serializable()
class SparseCrossentropyWithLabelSmoothing(tf.keras.losses.Loss):
    def __init__(self, num_classes, label_smoothing=0.0, name='sparse_ce_with_label_smoothing'):
        super().__init__(name=name)
        self.num_classes = int(num_classes)
        self.label_smoothing = float(label_smoothing)

    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_true = tf.one_hot(y_true, depth=self.num_classes, dtype=y_pred.dtype)
        if self.label_smoothing > 0:
            smooth = tf.cast(self.label_smoothing, y_pred.dtype)
            y_true = y_true * (1.0 - smooth) + smooth / tf.cast(self.num_classes, y_pred.dtype)
        return tf.keras.losses.categorical_crossentropy(y_true, y_pred)

    def get_config(self):
        config = super().get_config()
        config.update({'num_classes': self.num_classes, 'label_smoothing': self.label_smoothing})
        return config


def build_optimizer():
    adamw_cls = getattr(tf.keras.optimizers, 'AdamW', None)
    if adamw_cls is None:
        print('AdamW unavailable; fallback to Adam.')
        return tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    return adamw_cls(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


def set_batchnorm_momentum(layer, momentum=BN_MOMENTUM):
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.momentum = momentum
    if hasattr(layer, 'layers'):
        for sublayer in layer.layers:
            set_batchnorm_momentum(sublayer, momentum)


def build_model(num_classes):
    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3), name='image')
    x = tf.keras.layers.RandomFlip('horizontal')(inputs)
    x = tf.keras.layers.RandomTranslation(0.06, 0.06)(x)
    x = tf.keras.layers.RandomRotation(0.05)(x)
    x = tf.keras.layers.RandomZoom(0.12)(x)
    x = tf.keras.layers.RandomContrast(0.12)(x)
    x = tf.keras.layers.Rescaling(2.0, offset=-1.0, name='scale_to_minus1_1')(x)

    base = tf.keras.applications.MobileNetV2(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights=None,
    )
    set_batchnorm_momentum(base)
    base.trainable = True

    x = base(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(DROPOUT_RATE)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs, name='MobileNetV2_FromScratch')
    set_batchnorm_momentum(model)
    model.compile(
        optimizer=build_optimizer(),
        loss=SparseCrossentropyWithLabelSmoothing(num_classes, LABEL_SMOOTHING),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')],
    )
    return model


with strategy.scope():
    model = build_model(num_classes)

model.summary()


## 8. Train


In [ ]:
best_model_path = MODEL_DIR / f'{FILE_PREFIX}_best.keras'
final_model_path = MODEL_DIR / f'{FILE_PREFIX}_final.keras'
history_path = ARTIFACT_DIR / f'{FILE_PREFIX}_history.csv'


def cosine_warmup_lr(epoch):
    if epoch < LR_WARMUP_EPOCHS:
        return LEARNING_RATE * float(epoch + 1) / float(max(1, LR_WARMUP_EPOCHS))
    progress = float(epoch - LR_WARMUP_EPOCHS) / float(max(1, EPOCHS - LR_WARMUP_EPOCHS))
    cosine = 0.5 * (1.0 + np.cos(np.pi * progress))
    return LEARNING_RATE * (LR_MIN_FACTOR + (1.0 - LR_MIN_FACTOR) * cosine)


callbacks = [
    tf.keras.callbacks.TerminateOnNaN(),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(best_model_path),
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.LearningRateScheduler(cosine_warmup_lr, verbose=0),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.CSVLogger(str(history_path)),
]

history = None
train_time = 0

if RUN_TRAINING:
    start = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
    )
    train_time = time.time() - start
    print(f'Training time: {train_time:.1f}s')
    model.save(str(final_model_path))
    print('Saved best model:', best_model_path)
    print('Saved final model:', final_model_path)
else:
    print('RUN_TRAINING=False, skip training.')


## 9. Training Curves


In [ ]:
def safe_filename(name):
    return name.lower().replace(' ', '_').replace('-', '_')


def plot_training_history(history, title):
    if history is None:
        print('No history to plot.')
        return
    hist = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(hist['loss'], label='train_loss')
    axes[0].plot(hist['val_loss'], label='val_loss')
    axes[0].set_title(f'{title} - Loss')
    axes[0].legend()
    axes[1].plot(hist['accuracy'], label='train_accuracy')
    axes[1].plot(hist['val_accuracy'], label='val_accuracy')
    axes[1].set_title(f'{title} - Accuracy')
    axes[1].legend()
    plt.tight_layout()
    figure_path = FIGURE_DIR / f'{safe_filename(title)}_training_curves.png'
    fig.savefig(figure_path, dpi=160, bbox_inches='tight')
    plt.show()
    print('Saved:', figure_path)


plot_training_history(history, MODEL_NAME)


## 10. Evaluation


In [ ]:
if best_model_path.exists():
    eval_model = tf.keras.models.load_model(str(best_model_path), compile=False)
    eval_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=SparseCrossentropyWithLabelSmoothing(num_classes, LABEL_SMOOTHING),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')],
    )
else:
    eval_model = model


def predict_dataframe(model, dataframe):
    ds = make_dataset(dataframe, training=False)
    probabilities = model.predict(ds, verbose=1)
    pred_ids = np.argmax(probabilities, axis=1)
    pred_df = dataframe[['image_path', 'label', 'label_id']].copy().reset_index(drop=True)
    pred_df['true_id'] = pred_df['label_id'].astype(int)
    pred_df['pred_id'] = pred_ids.astype(int)
    pred_df['pred_label'] = [id_to_label[int(i)] for i in pred_ids]
    pred_df['confidence'] = np.max(probabilities, axis=1)
    pred_df['correct'] = pred_df['true_id'] == pred_df['pred_id']
    return pred_df, probabilities


pred_df, probabilities = predict_dataframe(eval_model, test_df)
y_true = pred_df['true_id'].values
y_pred = pred_df['pred_id'].values

acc = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=list(range(num_classes)), average='macro', zero_division=0
)
print(f'Test accuracy: {acc:.4f}')
print(f'Macro precision: {precision:.4f} | recall: {recall:.4f} | f1: {f1:.4f}')

report_text = classification_report(
    y_true, y_pred, labels=list(range(num_classes)), target_names=class_names, zero_division=0
)
print(report_text)

report_dict = classification_report(
    y_true, y_pred, labels=list(range(num_classes)), target_names=class_names, zero_division=0, output_dict=True
)
pd.DataFrame(report_dict).transpose().to_csv(ARTIFACT_DIR / f'{FILE_PREFIX}_classification_report.csv')
pred_df.to_csv(ARTIFACT_DIR / f'{FILE_PREFIX}_test_predictions.csv', index=False)
np.save(ARTIFACT_DIR / f'{FILE_PREFIX}_test_probabilities.npy', probabilities)

cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
fig = plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f'{MODEL_NAME} - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
cm_path = FIGURE_DIR / f'{FILE_PREFIX}_confusion_matrix.png'
fig.savefig(cm_path, dpi=160, bbox_inches='tight')
plt.show()
print('Saved:', cm_path)



## 11. Prediction Visualization


In [ ]:
CORRECT_SAMPLES_PER_LABEL = 5
MAX_WRONG_IMAGES_PER_LABEL = 30


def render_prediction_grid(rows, title, max_cols=5):
    rows = rows.reset_index(drop=True)
    if rows.empty:
        print('No images:', title)
        return
    cols = min(max_cols, len(rows))
    n_rows = math.ceil(len(rows) / cols)
    fig, axes = plt.subplots(n_rows, cols, figsize=(cols * 3.2, n_rows * 3.5))
    axes = np.array(axes).reshape(-1)
    for ax, (_, row) in zip(axes, rows.iterrows()):
        img = Image.open(row['image_path']).convert('RGB')
        ax.imshow(img)
        ax.set_title(f"true: {row['label']}\npred: {row['pred_label']}\nconf: {row['confidence']:.2f}", fontsize=9)
        ax.axis('off')
    for ax in axes[len(rows):]:
        ax.axis('off')
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


selected_correct = []
for label in class_names:
    part = pred_df[(pred_df['label'] == label) & (pred_df['correct'])]
    if not part.empty:
        selected_correct.append(part.sample(min(CORRECT_SAMPLES_PER_LABEL, len(part)), random_state=SEED))
if selected_correct:
    correct_df = pd.concat(selected_correct, ignore_index=True)
    correct_df.to_csv(ARTIFACT_DIR / f'{FILE_PREFIX}_correct_examples.csv', index=False)
    render_prediction_grid(correct_df, f'{MODEL_NAME} - Correct examples')

wrong_df = pred_df[~pred_df['correct']].copy()
wrong_df.to_csv(ARTIFACT_DIR / f'{FILE_PREFIX}_wrong_predictions.csv', index=False)
if wrong_df.empty:
    print('No wrong predictions.')
else:
    display(wrong_df.groupby(['label', 'pred_label']).size().reset_index(name='count').sort_values(['label', 'count'], ascending=[True, False]))
    render_prediction_grid(wrong_df.sort_values('confidence', ascending=False).head(MAX_WRONG_IMAGES_PER_LABEL), f'{MODEL_NAME} - Wrong predictions')


## 12. Saved Artifacts


In [ ]:
artifacts = sorted([p for p in ARTIFACT_DIR.rglob('*') if p.is_file()])
if artifacts:
    display(pd.DataFrame({
        'artifact_path': [str(p) for p in artifacts],
        'size_mb': [p.stat().st_size / (1024 * 1024) for p in artifacts],
    }).round({'size_mb': 3}))
else:
    print('No artifacts yet.')
